# 01 — TiNA Data Understanding

**Goal:** verify the source schema, time coverage, missingness, and maintenance/alarm distributions before preprocessing.

The official source is [ari-dasci/OD-TINA](https://github.com/ari-dasci/OD-TINA). The dataset is too large to load eagerly on a typical laptop, so this notebook starts with a bounded preview. Do not commit source data or notebook outputs containing large tables.

## Reproducibility record

Verified Week 2 pilot record:

- Source: `ari-dasci/OD-TINA` commit `927b1584ea4aeac2ac7daa084a82f899b114419b`; downloaded 2026-09-21.
- Local file: `data/raw/tina_sample_2017-09-08_to_10.parquet`; SHA-256 `19365a40e99083c37c9909f25b281caad6e47430c52b255c4c24b0db874db947`.
- Selected period: 2017-09-08 00:00:00 through 2017-09-10 23:59:59 UTC. This contiguous three-day window fits in laptop memory and contains maintenance and alarm activity.
- Analyst: PROJ 613 project team (Ganesh Mishra, Lakshya Sharma, and Sagar Patel).

In [1]:
from pathlib import Path
import os
import sys

import matplotlib.pyplot as plt
import polars as pl
import seaborn as sns

PROJECT_ROOT = Path.cwd().resolve().parent if Path.cwd().name == 'notebooks' else Path.cwd().resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.data_utils import resolve_data_path, scan_tina, validate_event_columns

DATA_PATH = resolve_data_path()
PREVIEW_ROWS = int(os.environ.get('TINA_PREVIEW_ROWS', '300000'))
print(f'Data path: {DATA_PATH}')
print(f'Bounded preview rows: {PREVIEW_ROWS:,}')

Data path: /Users/ganeshmishra/Documents/Codex/2026-09-19/referenced-chatgpt-conversation-this-is-an/outputs/ai-based-predictive-maintenance-mining-equipment/data/raw/tina_sample_2017-09-06.parquet
Bounded preview rows: 300,000


## Load lazily and validate

The preview supports quick iteration. Final claims must be rerun against a documented, event-preserving time period rather than an arbitrary row sample.

In [2]:
lazy = scan_tina(DATA_PATH)
validate_event_columns(lazy)
schema = lazy.collect_schema()
print(f'Columns: {len(schema.names())}')
schema

Columns: 109


Schema([('FEATURE0', Float64),
        ('FEATURE1', Float64),
        ('FEATURE2', Float64),
        ('FEATURE3', Float64),
        ('FEATURE4', Float64),
        ('FEATURE5', Float64),
        ('FEATURE6', Float64),
        ('FEATURE7', Float64),
        ('FEATURE8', Float64),
        ('FEATURE9', Float64),
        ('FEATURE10', Float64),
        ('FEATURE11', Float64),
        ('FEATURE12', Float64),
        ('FEATURE13', Float64),
        ('FEATURE14', Float64),
        ('FEATURE15', Float64),
        ('FEATURE16', Float64),
        ('FEATURE17', Float64),
        ('FEATURE18', Float64),
        ('FEATURE19', Float64),
        ('FEATURE20', Float64),
        ('FEATURE21', Float64),
        ('FEATURE22', Float64),
        ('FEATURE23', Float64),
        ('FEATURE24', Float64),
        ('FEATURE25', Float64),
        ('FEATURE26', Float64),
        ('FEATURE27', Float64),
        ('FEATURE28', Float64),
        ('FEATURE29', Float64),
        ('FEATURE30', Float64),
        ('FEATURE3

In [3]:
preview = lazy.head(PREVIEW_ROWS).collect()
print(f'Preview shape: {preview.shape}')
preview.head()

Preview shape: (74026, 109)


FEATURE0,FEATURE1,FEATURE2,FEATURE3,FEATURE4,FEATURE5,FEATURE6,FEATURE7,FEATURE8,FEATURE9,FEATURE10,FEATURE11,FEATURE12,FEATURE13,FEATURE14,FEATURE15,FEATURE16,FEATURE17,FEATURE18,FEATURE19,FEATURE20,FEATURE21,FEATURE22,FEATURE23,FEATURE24,FEATURE25,FEATURE26,FEATURE27,FEATURE28,FEATURE29,FEATURE30,FEATURE31,FEATURE32,FEATURE33,FEATURE34,FEATURE35,FEATURE36,…,FEATURE72,FEATURE73,FEATURE74,FEATURE75,FEATURE76,FEATURE77,FEATURE78,FEATURE79,FEATURE80,FEATURE81,FEATURE82,FEATURE83,FEATURE84,FEATURE85,FEATURE86,FEATURE87,FEATURE88,FEATURE89,FEATURE90,FEATURE91,FEATURE92,FEATURE93,FEATURE94,FEATURE95,FEATURE96,FEATURE97,FEATURE98,FEATURE99,FEATURE100,FEATURE101,FEATURE102,FEATURE103,FEATURE104,m_id,m_subid,alarms,unix
f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,…,f64,f64,f64,f64,str,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,str,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,str,str,str,i64
0.28961,0.185029,-0.173751,0.010513,-1.352401,0.144183,0.129125,0.001384,-0.065001,-0.615463,0.085374,0.696724,0.302551,-0.339677,-0.97203,-1.352401,0.560856,0.623755,0.13021,-0.490904,0.010176,0.449501,0.548263,-0.373079,0.11935,-0.973038,-0.047979,-0.000434,0.128138,0.077555,0.465855,0.429645,-0.042964,-0.03918,-0.033683,0.108803,0.048102,…,-0.422735,0.921633,0.91392,-1.597979,"""MI2""",-0.045164,0.373507,2.047508,1.157481,0.308048,1.994843,0.902188,0.083852,0.390231,1.364555,"""LS37""",-1.649163,-1.606435,2.489893,0.914034,0.635845,1.802994,-1.65369,3.514155,-1.268983,1.683808,1.658084,1.134894,0.789802,-0.945786,-1.281975,0.707066,-1.643781,"""none""","""none""","""none""",1504668520
0.28961,0.185029,-0.172079,0.010513,0.739425,0.144183,0.129125,0.001384,-0.065001,-0.615463,0.085374,0.696724,0.302551,-0.339677,-0.97203,0.739426,0.680813,0.699133,0.202115,-0.490904,0.010176,0.449501,0.548263,-0.373079,0.11935,-0.973038,-1.366509,-1.424929,-1.375139,-1.417426,-1.205329,0.429645,-1.392387,-1.429305,-1.347578,-1.334512,-1.376077,…,-0.422735,0.937631,0.929836,-1.597979,"""MI2""",-0.045164,0.373507,2.047508,1.157481,0.308048,1.994843,0.902188,0.083852,0.390231,1.364555,"""LS37""",-1.649163,-1.606435,2.489893,0.914034,0.635845,1.802994,-1.65369,3.514155,-1.268983,1.683808,1.658084,1.134894,0.789802,-0.945786,-1.281975,0.707066,-1.643781,"""none""","""none""","""A97""",1504668964
0.28961,0.185029,-0.172079,0.010513,0.739425,0.144183,0.085421,0.001384,-0.065001,-0.615463,0.085374,0.696724,0.302551,-0.339677,-0.97203,0.739426,0.560856,0.699133,0.202115,-0.490904,0.010176,0.449501,0.455782,-0.420096,0.11935,-0.973038,-1.366509,-1.424929,-1.375139,-1.417426,-1.205329,0.429645,-1.392387,-1.429305,-1.347578,-1.334512,-1.376077,…,-0.422735,0.988534,0.980478,-1.597979,"""MI2""",-0.045164,0.373507,2.047508,1.157481,0.308048,1.994843,0.902188,0.083852,0.390231,1.364555,"""LS37""",-1.649163,-1.606435,2.489893,0.914034,0.635845,1.802994,-1.65369,3.514155,-1.268983,1.683808,1.658084,1.134894,0.789802,-0.945786,-1.281975,0.707066,-1.643781,"""none""","""none""","""A97""",1504669121
0.28961,0.173114,-0.172079,0.010513,0.739425,0.144183,0.085421,0.001384,-0.065001,-0.615463,0.085374,0.696724,0.302551,-0.339677,-0.97203,0.739426,0.560856,0.699133,0.13021,-0.490904,0.010176,0.449501,0.455782,-0.420096,0.11935,-0.973038,-1.366509,-1.424929,-1.375139,-1.417426,-1.205329,0.429645,-1.392387,-1.429305,-1.347578,-1.334512,-1.376077,…,-0.422735,0.976414,0.96842,-1.597979,"""MI2""",-0.045164,0.373507,2.047508,1.157481,0.308048,1.994843,0.902188,0.083852,0.390231,1.364555,"""LS37""",-1.649163,-1.606435,2.489893,0.914034,0.635845,1.802994,-1.65369,3.514155,-1.268983,1.683808,1.658084,1.134894,0.789802,-0.945786,-1.281975,0.707066,-1.643781,"""none""","""none""","""A97""",1504669164
0.34802,0.173114,-0.172079,0.001997,0.739425,0.144183,0.085421,0.001384,-0.026499,-0.615463,0.085374,0.696724,0.302551,-0.339677,-0.97203,0.739426,0.560856,0.774511,0.202115,-0.49

## Time field and ordering

Confirm the real timestamp column manually if it is not detected. Never build lags, rolling features, or train/test splits until ascending time order is verified.

In [4]:
TIME_CANDIDATES = ('unix', 'timestamp', 'datetime', 'date', 'time', 'Date', 'Time')
time_col = next((name for name in TIME_CANDIDATES if name in preview.columns), None)
print(f'Detected time column: {time_col!r}')
if time_col:
    print(preview.select(pl.col(time_col).min().alias('min_time'), pl.col(time_col).max().alias('max_time')))
    print(f'Ascending in preview: {preview[time_col].is_sorted()}')
else:
    print('Set time_col to the verified timestamp field before continuing to time-based analysis.')

Detected time column: 'unix'
shape: (1, 2)
┌────────────┬────────────┐
│ min_time   ┆ max_time   │
│ ---        ┆ ---        │
│ i64        ┆ i64        │
╞════════════╪════════════╡
│ 1504668374 ┆ 1504742399 │
└────────────┴────────────┘
Ascending in preview: False


## Missingness and constant fields

In [5]:
quality = pl.DataFrame({
    'column': preview.columns,
    'dtype': [str(dtype) for dtype in preview.dtypes],
    'null_count': [preview[column].null_count() for column in preview.columns],
    'n_unique': [preview[column].n_unique() for column in preview.columns],
}).with_columns((pl.col('null_count') / preview.height * 100).alias('null_pct'))
quality.sort('null_pct', descending=True).head(25)

column,dtype,null_count,n_unique,null_pct
str,str,i64,i64,f64
"""FEATURE0""","""Float64""",0,3,0.0
"""FEATURE1""","""Float64""",0,6,0.0
"""FEATURE2""","""Float64""",0,3,0.0
"""FEATURE3""","""Float64""",0,5,0.0
"""FEATURE4""","""Float64""",0,2,0.0
…,…,…,…,…
"""FEATURE20""","""Float64""",0,2,0.0
"""FEATURE21""","""Float64""",0,2,0.0
"""FEATURE22""","""Float64""",0,3,0.0


## Maintenance and alarm distributions

In [6]:
for column in ('m_id', 'm_subid', 'alarms'):
    display(
        preview.group_by(column)
        .len(name='rows')
        .with_columns((pl.col('rows') / preview.height * 100).alias('pct'))
        .sort('rows', descending=True)
        .head(20)
    )

m_id,rows,pct
str,u32,f64
"""none""",74026,100.0


m_subid,rows,pct
str,u32,f64
"""none""",74026,100.0


alarms,rows,pct
str,u32,f64
"""none""",72145,97.459001
"""A97""",1802,2.43428
"""A5""",74,0.099965
"""A96""",4,0.005404
"""A95""",1,0.001351


## Initial decisions

Verified findings and provisional decisions:

- Verified timestamp and sampling interval: `unix` in UTC seconds; 259,200 unique, ascending timestamps at an exact one-second cadence.
- Data-quality issues requiring treatment: no null, NaN, or infinite values in the pilot; eight constant sensor columns require reassessment on broader data; categorical sensors and event codes require explicit encoding; class imbalance is material.
- Maintenance/alarm codes present: `m_id` M4/M9; `m_subid` MS3/MS5/MS6; alarms A5/A12/A13/A23/A34/A35/A95/A96/A97/A111/A115/A116/A118/A128.
- Event-preserving subset and memory estimate: 2017-09-08 through 2017-09-10, 259,200 rows, 4,994 maintenance rows, 26,719 alarm rows, about 2.5 MB Parquet and 220.7 MB in memory.
- Candidate prediction horizons: provisional 30 minutes, 1 hour, and 4 hours; finalize during Week 6 target design.
- Recovery/exclusion rule: exclude rows with active maintenance; test a provisional 30-minute post-maintenance recovery window during Week 3 event-duration analysis.